In [ ]:
# =============================================================================
# SETUP (if code up to token counting has not been run yet)
# =============================================================================

# Uncomment if needed
# %pip install -U openai pandas tqdm

import json
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# =============================================================================
# CONFIGURATION
# =============================================================================

from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

MODEL = "gpt-4o"
# MODEL = "gpt-4.1-mini"   # cheaper alternative

INPUT_CSV = "data/solar_articles.csv"

OUTPUT_JSON = "graph/solar_graphs.json"
OUTPUT_NODES = "graph/nodes.csv"
OUTPUT_EDGES = "graph/edges.csv"

SAVE_CHECKPOINT_EVERY = 25

# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(INPUT_CSV)

print(f"Loaded {len(df)} articles")

# Optional:
# df = df.head(100)

Loaded 5374 articles


In [2]:
# =============================================================================
# COUNT TOKENS IN ARTICLES
# =============================================================================

# Install if needed:
# %pip install tiktoken

import tiktoken
import pandas as pd

# Use the tokenizer closest to your model
encoding = tiktoken.encoding_for_model(MODEL)


def count_tokens(text):
    """
    Count tokens in a string using the model tokenizer.
    """
    if pd.isna(text):
        return 0
    
    return len(
        encoding.encode(str(text))
    )


# Count tokens in article text
df["token_count"] = df["text"].apply(count_tokens)


print(df[[
    "title",
    "token_count"
]].head())


print("\nToken statistics:")
print(df["token_count"].describe())


                                               title  token_count
0       UPDATE: You asked, the White House responds!          382
1  'Starry Night' bike path illuminates the groun...          210
2         Assessing the Value of Small Wind Turbines           32
3  How 24 Sussex Drive can become the greenest of...         1444
4  What Electric Grid Operators Want: Good Wind E...          732

Token statistics:
count     5374.000000
mean      1241.775214
std       1545.379231
min         18.000000
25%        583.000000
50%        941.500000
75%       1344.500000
max      42954.000000
Name: token_count, dtype: float64


In [3]:
import uuid

# Create stable article IDs
df["article_id"] = [
    f"article_{i:06d}"
    for i in range(len(df))
]

# Save updated CSV
df.to_csv(
    "solar_articles_with_tokens.csv",
    index=False
)

In [4]:
# =============================================================================
# SETUP (if above code has been run previously)
# =============================================================================

import json
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from dotenv import load_dotenv
import os
import tiktoken
import pandas as pd

# Use the tokenizer closest to your model
encoding = tiktoken.encoding_for_model(MODEL)

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

MODEL = "gpt-4o"
# MODEL = "gpt-4.1-mini"   # cheaper alternative

INPUT_CSV = "data/solar_articles_w_tokens.csv"

OUTPUT_JSON = "graph/solar_graphs.json"
OUTPUT_NODES = "graph/nodes.csv"
OUTPUT_EDGES = "graph/edges.csv"

SAVE_CHECKPOINT_EVERY = 25

df = pd.read_csv(INPUT_CSV)

print(f"Loaded {len(df)} articles")

# Optional:
# df = df.head(100)

Loaded 5374 articles


In [5]:
from pathlib import Path

SYSTEM_PROMPT = Path("prompts/system_prompt.txt").read_text()

USER_PROMPT = Path("prompts/user_prompt.txt").read_text()

with open("prompts/solar_graph_schema.json") as f:
    SOLAR_GRAPH_SCHEMA = json.load(f)

In [6]:
# Optional: count the system prompt size

print(len(encoding.encode(SYSTEM_PROMPT)), "tokens in system prompt")

1773 tokens in system prompt


In [7]:
# =============================================================================
# EXTRACTION FUNCTION
# =============================================================================

def extract_article(row):

    prompt = USER_PROMPT.format(
        article_id=["article_id"],
        publication_date=row["date"],
        source=row["archive"],
        title=row["title"],
        summary=row.get("summary", ""),
        text=row["text"],
    )

    response = client.chat.completions.create(

        model=MODEL,

        temperature=0,

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        response_format={
            "type": "json_schema",
            "json_schema": SOLAR_GRAPH_SCHEMA
        }

    )

    return json.loads(response.choices[0].message.content)

In [8]:
# =============================================================================
# TEST EXTRACTION ON A SINGLE ARTICLE
# =============================================================================

# Change this index to test a different article
TEST_ARTICLE_IDX = 0

row = df.iloc[TEST_ARTICLE_IDX]

print(f"Testing article index {TEST_ARTICLE_IDX}")
print(f"Title: {row['title']}")
print(f"Article ID: {row.name}")

single_result = extract_article(row)

print("\nUnrelated:", single_result["unrelated"])
print("Nodes:", len(single_result["nodes"]))
print("Edges:", len(single_result["edges"]))

print("\nPreview of extracted JSON:")
print(json.dumps(single_result, indent=2))


Testing article index 0
Title: UPDATE: You asked, the White House responds!
Article ID: 0

Unrelated: True
Nodes: 0
Edges: 0

Preview of extracted JSON:
{
  "article_id": "article_id",
  "publication_date": "2011082419",
  "source": "https://web.archive.org/web/2011082419id_/http://ireport.cnn.com/blogs/ireport-blog/2011/08/22/update-you-asked-the-white-house-responds",
  "unrelated": true,
  "nodes": [],
  "edges": []
}


In [9]:
# =============================================================================
# TEST EXTRACTION ON A SINGLE ARTICLE
# =============================================================================

# Change this index to test a different article
TEST_ARTICLE_IDX = 5

row = df.iloc[TEST_ARTICLE_IDX]

print(f"Testing article index {TEST_ARTICLE_IDX}")
print(f"Title: {row['title']}")
print(f"Article ID: {row.name}")

single_result = extract_article(row)

print("\nUnrelated:", single_result["unrelated"])
print("Nodes:", len(single_result["nodes"]))
print("Edges:", len(single_result["edges"]))

print("\nPreview of extracted JSON:")
print(json.dumps(single_result, indent=2))


Testing article index 5
Title: Tesla Charges into Home Battery Market Despite Challenges
Article ID: 5

Unrelated: False
Nodes: 11
Edges: 6

Preview of extracted JSON:
{
  "article_id": "article_id",
  "publication_date": "2015050119",
  "source": "https://web.archive.org/web/2015050119id_/http://time.com/3842996/tesla-elon-musk-solar-panel-batteries/",
  "unrelated": false,
  "nodes": [
    {
      "id": "company:tesla",
      "type": "Company",
      "properties": {
        "name": "Tesla",
        "ticker": null,
        "headquarters": null,
        "country": null,
        "government_level": null,
        "organization_type": null,
        "policy_type": null,
        "jurisdiction": null,
        "enactment_year": null,
        "capacity": null,
        "location": null,
        "time_period": null,
        "location_type": null,
        "role_in_solar_energy_space": null
      },
      "mentions": [
        "Tesla",
        "Tesla Motors"
      ]
    },
    {
      "id": "perso

In [ ]:
# =============================================================================
# RUN EXTRACTION
# =============================================================================

results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    success = False

    for attempt in range(3):

        try:

            result = extract_article(row)

            results.append(result)

            success = True

            break

        except Exception as e:

            print(f"Row {idx} failed (attempt {attempt+1}): {e}")

            time.sleep(2)

    if not success:

        print(f"Skipping article {idx}")

    if len(results) % SAVE_CHECKPOINT_EVERY == 0:

        with open(OUTPUT_JSON, "w") as f:
            json.dump(results, f, indent=2)

In [ ]:
# =============================================================================
# SAVE FINAL JSON
# =============================================================================

with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} extracted graphs.")

In [ ]:
# =============================================================================
# FLATTEN NODES
# =============================================================================

nodes = []

for article in results:

    for node in article["nodes"]:

        node_copy = node.copy()

        node_copy["article_id"] = article["article_id"]

        nodes.append(node_copy)

nodes_df = pd.DataFrame(nodes)

nodes_df.to_csv(OUTPUT_NODES, index=False)

print(f"Saved {len(nodes_df)} nodes.")

In [ ]:
# =============================================================================
# FLATTEN EDGES
# =============================================================================

edges = []

for article in results:

    edges.extend(article["edges"])

edges_df = pd.DataFrame(edges)

edges_df.to_csv(OUTPUT_EDGES, index=False)

print(f"Saved {len(edges_df)} edges.")

In [ ]:
# =============================================================================
# OPTIONAL ANALYSIS
# =============================================================================

print(nodes_df.head())

print(edges_df.head())

print(edges_df["relationship"].value_counts())

print(nodes_df["type"].value_counts())

# Example:
#
# company_nodes = nodes_df[nodes_df["type"] == "Company"]
#
# predictions = edges_df[
#     edges_df["relationship"] == "PREDICTS"
# ]
#
# partnerships = edges_df[
#     edges_df["relationship"] == "PARTNERS_WITH"
# ]
#
# unrelated_articles = [
#     r for r in results if r["unrelated"]
# ]
#
# print(len(unrelated_articles))
